# 03 - Backtesting: RSI Momentum + Filtro de Tendencia

**Capítulo**: 03 - Momentum RSI

**Objetivo**: Backtest de RSI con/sin filtro de tendencia, comparación y optimización de umbrales.

---

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
from backtesting import Backtest, Strategy
from backtesting.test import SMA

from curso.lib.data import download_historical
from curso.lib.backtest import run_backtest, extract_metrics, metrics_to_dataframe, compare_strategies
from curso.lib.reporting import plot_equity_curve, plot_drawdown, print_metrics_table, plot_comparison

import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
print('Setup completado ✓')

## 1. Datos

In [ ]:
TICKER = 'AAPL'
df = download_historical(TICKER)
print(f'{TICKER}: {len(df)} registros')

## 2. Estrategia RSI (sin filtro)

In [ ]:
def RSI_indicator(values, n):
    """RSI compatible con backtesting.py."""
    s = pd.Series(values)
    delta = s.diff()
    gain = delta.where(delta > 0, 0).rolling(n).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(n).mean()
    rs = gain / loss
    return (100 - 100 / (1 + rs)).values


class RSIStrategy(Strategy):
    rsi_period = 14
    rsi_lower = 30
    rsi_upper = 70
    stop_loss_pct = 8
    
    def init(self):
        self.rsi = self.I(RSI_indicator, self.data.Close, self.rsi_period)
    
    def next(self):
        if not self.position:
            if self.rsi[-1] < self.rsi_lower:
                price = self.data.Close[-1]
                self.buy(sl=price * (1 - self.stop_loss_pct/100))
        else:
            if self.rsi[-1] > self.rsi_upper:
                self.position.close()

## 3. Estrategia RSI con filtro de tendencia

In [ ]:
class RSITrendStrategy(Strategy):
    rsi_period = 14
    rsi_lower = 30
    rsi_upper = 70
    sma_trend = 200
    stop_loss_pct = 8
    
    def init(self):
        self.rsi = self.I(RSI_indicator, self.data.Close, self.rsi_period)
        self.sma = self.I(SMA, self.data.Close, self.sma_trend)
    
    def next(self):
        price = self.data.Close[-1]
        in_uptrend = price > self.sma[-1]
        
        if not self.position:
            if self.rsi[-1] < self.rsi_lower and in_uptrend:
                self.buy(sl=price * (1 - self.stop_loss_pct/100))
        else:
            if self.rsi[-1] > self.rsi_upper:
                self.position.close()

## 4. Backtest comparativo

In [ ]:
# Sin filtro
stats_nf, _ = run_backtest(df, RSIStrategy)
print('=== RSI Sin Filtro ===')
print_metrics_table(metrics_to_dataframe(extract_metrics(stats_nf)))

# Con filtro
stats_wf, _ = run_backtest(df, RSITrendStrategy)
print('\n=== RSI Con Filtro de Tendencia ===')
print_metrics_table(metrics_to_dataframe(extract_metrics(stats_wf)))

## 5. Visualización comparativa

In [ ]:
comparison = compare_strategies(
    {'RSI Simple': stats_nf, 'RSI + Trend Filter': stats_wf}
)
plot_comparison(comparison)
plt.show()

## 6. Conclusiones

In [ ]:
print('''
CONCLUSIONES
============
1. El filtro de tendencia [mejora/empeora] el rendimiento
2. Win rate: [comparar ambas]
3. Max drawdown: [comparar ambas]
4. DECISIÓN: [aprobar / iterar con nuevos umbrales]
''')